# Predictive Maintenance Using NASA C-MAPSS

## Phase 3 — Machine Learning

Train a Random Forest regression model to predict Remaining Useful Life (RUL), evaluate its performance, and track experiments using MLflow.

In [0]:
from pyspark.sql import functions as F

schema = "workspace.predictive_maintenance_rul"

# Load Gold feature tables
gold_train = spark.table(f"{schema}.gold_train_features")
gold_test = spark.table(f"{schema}.gold_test_features")
test_rul = spark.table(f"{schema}.silver_test_rul")

print("Training rows:", gold_train.count())
print("Training columns:", len(gold_train.columns))

print("Test rows:", gold_test.count())
print("Test columns:", len(gold_test.columns))

print("Official test RUL labels:", test_rul.count())

display(gold_train.limit(5))

Training rows: 20631
Training columns: 70
Test rows: 13096
Test columns: 68
Official test RUL labels: 100


unit_number,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,max_cycle,RUL,sensor_1_rolling_avg_3,sensor_1_change,sensor_2_rolling_avg_3,sensor_2_change,sensor_3_rolling_avg_3,sensor_3_change,sensor_4_rolling_avg_3,sensor_4_change,sensor_5_rolling_avg_3,sensor_5_change,sensor_6_rolling_avg_3,sensor_6_change,sensor_7_rolling_avg_3,sensor_7_change,sensor_8_rolling_avg_3,sensor_8_change,sensor_9_rolling_avg_3,sensor_9_change,sensor_10_rolling_avg_3,sensor_10_change,sensor_11_rolling_avg_3,sensor_11_change,sensor_12_rolling_avg_3,sensor_12_change,sensor_13_rolling_avg_3,sensor_13_change,sensor_14_rolling_avg_3,sensor_14_change,sensor_15_rolling_avg_3,sensor_15_change,sensor_16_rolling_avg_3,sensor_16_change,sensor_17_rolling_avg_3,sensor_17_change,sensor_18_rolling_avg_3,sensor_18_change,sensor_19_rolling_avg_3,sensor_19_change,sensor_20_rolling_avg_3,sensor_20_change,sensor_21_rolling_avg_3,sensor_21_change
1,1,-7.0E-4,-4.0E-4,100.0,518.67,641.82,1589.7,1400.6,14.62,21.61,554.36,2388.06,9046.19,1.3,47.47,521.66,2388.02,8138.62,8.4195,0.03,392.0,2388.0,100.0,39.06,23.419,192,191.0,518.67,null,641.82,null,1589.7,null,1400.6,null,14.62,null,21.61,null,554.36,null,2388.06,null,9046.19,null,1.3,null,47.47,null,521.66,null,2388.02,null,8138.62,null,8.4195,null,0.03,null,392.0,null,2388.0,null,100.0,null,39.06,null,23.419,null
1,2,0.0019,-3.0E-4,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,553.75,2388.04,9044.07,1.3,47.49,522.28,2388.07,8131.49,8.4318,0.03,392.0,2388.0,100.0,39.0,23.4236,192,190.0,518.67,0.0,641.985,0.32999999999992724,1590.76,2.119999999999891,1401.87,2.540000000000191,14.62,0.0,21.61,0.0,554.0550000000001,-0.6100000000000136,2388.05,-0.01999999999998181,9045.130000000001,-2.1200000000008004,1.3,0.0,47.480000000000004,0.020000000000003126,521.97,0.6200000000000045,2388.045,0.0500000000001819,8135.055,-7.130000000000109,8.425650000000001,0.012300000000001532,0.03,0.0,392.0,0.0,2388.0,0.0,100.0,0.0,39.03,-0.060000000000002274,23.421300000000002,0.0045999999999999375
1,3,-0.0043,3.0E-4,100.0,518.67,642.35,1587.99,1404.2,14.62,21.61,554.26,2388.08,9052.94,1.3,47.27,522.42,2388.03,8133.23,8.4178,0.03,390.0,2388.0,100.0,38.95,23.3442,192,189.0,518.67,0.0,642.1066666666667,0.20000000000004547,1589.8366666666668,-3.8299999999999272,1402.6466666666665,1.0599999999999454,14.62,0.0,21.61,0.0,554.1233333333333,0.5099999999999909,2388.06,0.03999999999996362,9047.733333333335,8.8700000000008,1.3,0.0,47.410000000000004,-0.21999999999999886,522.12,0.13999999999998636,2388.0400000000004,-0.03999999999996362,8134.446666666667,1.7399999999997817,8.423033333333334,-0.014000000000001123,0.03,0.0,391.3333333333333,-2.0,2388.0,0.0,100.0,0.0,39.00333333333334,-0.04999999999999716,23.3956,-0.07939999999999969
1,4,7.0E-4,0.0,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,554.45,2388.11,9049.48,1.3,47.13,522.86,2388.08,8133.83,8.3682,0.03,392.0,2388.0,100.0,38.88,23.3739,192,188.0,518.67,0.0,642.2833333333333,0.0,1587.5333333333335,-5.2000000000000455,1403.07,-2.3300000000001546,14.62,0.0,21.61,0.0,554.1533333333333,0.19000000000005457,2388.0766666666664,0.03000000000020009,9048.83,-3.460000000000946,1.3,0.0,47.296666666666674,-0.14000000000000057,522.52,0.44000000000005457,2388.06,0.04999999999972715,8132.849999999999,0.6000000000003638,8.405933333333335,-0.049599999999999866,0.03,0.0,391.3333333333333,2.0,2388.0,0.0,100.0,0.0,38.943333333333335,-0.07000000000000028,23.380566666666667,0.029699999999998283
1,5,-0.0019,-2.0E-4,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,554.0,2388.06,9055.15,1.3,47.28,522.19,2388.04,8133.8,8.4294,0.03,393.0,2388.0,100.0,38.9,23.4044,192,187.0,518.67,0.0,642.3566666666667,0.01999999999998181,1584.543333333333,0.05999999999994543,1404.0966666666666,4.350000000000136,14.62,0.0,21.61,0.0,554.2366666666667,-0.4500000

In [0]:
from pyspark.sql import functions as F

# Define input features
feature_cols = (
    [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
    + [f"sensor_{i}_rolling_avg_3" for i in range(1, 22)]
    + [f"sensor_{i}_change" for i in range(1, 22)]
)

print("Number of input features:", len(feature_cols))

# Get unique engine IDs and split them reproducibly
engine_ids = [
    row["unit_number"]
    for row in gold_train.select("unit_number").distinct().collect()
]

engine_ids = sorted(engine_ids)

train_engine_ids = engine_ids[:80]
validation_engine_ids = engine_ids[80:]

# Split all cycles from each engine into the same group
train_df = gold_train.filter(
    F.col("unit_number").isin(train_engine_ids)
)

validation_df = gold_train.filter(
    F.col("unit_number").isin(validation_engine_ids)
)

print("Training engines:", len(train_engine_ids))
print("Validation engines:", len(validation_engine_ids))
print("Training rows:", train_df.count())
print("Validation rows:", validation_df.count())

# Check target and feature columns
print("Target column: RUL")
print("Feature columns:", len(feature_cols))

Number of input features: 66
Training engines: 80
Validation engines: 20
Training rows: 16138
Validation rows: 4493
Target column: RUL
Feature columns: 66


In [0]:
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Keep only rows with valid target values
train_model_df = train_df.filter(F.col("RUL").isNotNull())
validation_model_df = validation_df.filter(F.col("RUL").isNotNull())

# Replace missing feature values using training data medians
imputer = Imputer(
    inputCols=feature_cols,
    outputCols=[f"{c}_imputed" for c in feature_cols],
    strategy="median"
)

imputed_features = [f"{c}_imputed" for c in feature_cols]

assembler = VectorAssembler(
    inputCols=imputed_features,
    outputCol="features",
    handleInvalid="keep"
)

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="RUL",
    predictionCol="predicted_RUL",
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline = Pipeline(stages=[imputer, assembler, rf])

# Train and evaluate with MLflow tracking
mlflow.set_experiment("/Shared/PredictiveMaintenance_RUL_Rebuild")

with mlflow.start_run(run_name="NASA_FD001_RandomForest"):
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("feature_count", len(feature_cols))
    mlflow.log_param("training_engines", len(train_engine_ids))
    mlflow.log_param("validation_engines", len(validation_engine_ids))

    model = pipeline.fit(train_model_df)

    predictions = model.transform(validation_model_df)

    mae = RegressionEvaluator(
        labelCol="RUL",
        predictionCol="predicted_RUL",
        metricName="mae"
    ).evaluate(predictions)

    rmse = RegressionEvaluator(
        labelCol="RUL",
        predictionCol="predicted_RUL",
        metricName="rmse"
    ).evaluate(predictions)

    r2 = RegressionEvaluator(
        labelCol="RUL",
        predictionCol="predicted_RUL",
        metricName="r2"
    ).evaluate(predictions)

    mlflow.log_metric("validation_MAE", mae)
    mlflow.log_metric("validation_RMSE", rmse)
    mlflow.log_metric("validation_R2", r2)

    print(f"Validation MAE:  {mae:.2f} cycles")
    print(f"Validation RMSE: {rmse:.2f} cycles")
    print(f"Validation R²:   {r2:.4f}")
    # Display the MLflow run ID for the trained model
    print("MLflow Run ID:", mlflow.last_active_run().info.run_id)

display(
    predictions.select(
        "unit_number", "cycle", "RUL", "predicted_RUL"
    ).limit(10)
)

Validation MAE:  35.52 cycles
Validation RMSE: 50.25 cycles
Validation R²:   0.5795
MLflow Run ID: 299f5f156b2a48b6b4d9d90d41b2b943


unit_number,cycle,RUL,predicted_RUL
81,1,239.0,145.3176557618138
81,2,238.0,140.55179357412936
81,3,237.0,146.41293037028868
81,4,236.0,135.95643506420083
81,5,235.0,145.01570608447958
81,6,234.0,150.7581776158779
81,7,233.0,148.1944328892295
81,8,232.0,151.20243268940933
81,9,231.0,141.56951211180422
81,10,230.0,145.20899737392767


In [0]:
import os
import mlflow
import mlflow.spark

# Use a Unity Catalog Volume for Spark ML temporary files
os.environ["MLFLOW_DFS_TMP"] = (
    "/Volumes/workspace/default/cmapss_data/mlflow_tmp"
)

# Check that the trained model and run ID are available
assert model is not None, "Model is not available. Run the training cell first."
assert mlflow.last_active_run() is not None, (
    "No active MLflow run context is available."
)

print("Trained model is available.")
print("Next, we will save it to MLflow.")

Trained model is available.
Next, we will save it to MLflow.


In [0]:
import os
import mlflow
import mlflow.spark

# Temporary storage inside your existing Unity Catalog Volume
temp_path = "/Volumes/workspace/default/cmapss_data/mlflow_tmp"
os.makedirs(temp_path, exist_ok=True)

# Save the existing trained pipeline in a new MLflow run
with mlflow.start_run(run_name="NASA_FD001_RandomForest_Model_Saved"):
    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)
    mlflow.log_param("feature_count", len(feature_cols))

    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="random_forest_pipeline",
        dfs_tmpdir=temp_path
    )

    saved_run_id = mlflow.active_run().info.run_id

print("Model logging completed.")
print("Saved MLflow Run ID:", saved_run_id)
print("Model URI:", f"runs:/{saved_run_id}/random_forest_pipeline")

2026/10/26 14:24:51 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-438b7798-23f8-420a-afd4-fc/tmpq9av66gm/model, flavor: spark). Fall back to return ['pyspark==4.3.0']. Set logging level to DEBUG to see the full traceback. 
2026/10/26 14:24:51 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


Model logging completed.
Saved MLflow Run ID: 1f0d8c6176154f74a2dcc3269f8b3cd3
Model URI: runs:/1f0d8c6176154f74a2dcc3269f8b3cd3/random_forest_pipeline


In [0]:
import mlflow

saved_run = mlflow.get_run(
    "1f0d8c6176154f74a2dcc3269f8b3cd3"
)

print("Run status:", saved_run.info.status)

client = mlflow.tracking.MlflowClient()
artifacts = client.list_artifacts(
    "1f0d8c6176154f74a2dcc3269f8b3cd3"
)

print("Saved artifacts:")
for artifact in artifacts:
    print(artifact.path, "| directory:", artifact.is_dir)

Run status: FINISHED
Saved artifacts:
random_forest_pipeline | directory: True


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Find the last observed cycle for each test engine
test_last_cycles = (
    gold_test
    .groupBy("unit_number")
    .agg(F.max("cycle").alias("last_cycle"))
)

test_final = (
    gold_test
    .join(test_last_cycles, on="unit_number")
    .filter(F.col("cycle") == F.col("last_cycle"))
)

# Assign engine IDs to official RUL labels in file order.
# NASA FD001 labels are ordered by test engine number.
label_window = Window.orderBy(F.monotonically_increasing_id())

test_rul_numbered = (
    test_rul
    .withColumn("unit_number", F.row_number().over(label_window))
    .select(
        "unit_number",
        F.col("official_rul").alias("actual_RUL")
    )
)

print("Test engines:", test_final.select("unit_number").distinct().count())
print("Official labels:", test_rul_numbered.count())

# Predict RUL for each engine's final observed cycle
test_predictions = model.transform(test_final)

# Join predictions with the corresponding official labels
test_evaluation = (
    test_predictions
    .join(test_rul_numbered, on="unit_number", how="inner")
    .withColumn(
        "absolute_error",
        F.abs(F.col("actual_RUL") - F.col("predicted_RUL"))
    )
)

test_mae = test_evaluation.agg(
    F.avg("absolute_error").alias("MAE")
).first()["MAE"]

test_rmse = test_evaluation.select(
    F.pow(F.col("actual_RUL") - F.col("predicted_RUL"), 2)
    .alias("squared_error")
).agg(
    F.sqrt(F.avg("squared_error")).alias("RMSE")
).first()["RMSE"]

print(f"Official test MAE:  {test_mae:.2f} cycles")
print(f"Official test RMSE: {test_rmse:.2f} cycles")
print("Engines evaluated:", test_evaluation.count())

display(
    test_evaluation.select(
        "unit_number",
        "last_cycle",
        "actual_RUL",
        "predicted_RUL",
        "absolute_error"
    ).orderBy("unit_number")
)

Test engines: 100


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Official labels: 100
Official test MAE:  20.89 cycles
Official test RMSE: 28.58 cycles
Engines evaluated: 100


unit_number,last_cycle,actual_RUL,predicted_RUL,absolute_error
1,31,112,151.80743830160208,39.807438301602076
2,49,98,140.7000757291631,42.7000757291631
3,126,69,58.62785425728279,10.372145742717208
4,106,82,108.70957947059729,26.70957947059729
5,98,91,118.32314453217248,27.323144532172478
6,105,93,137.88848911655674,44.88848911655674
7,160,91,116.08539250648928,25.085392506489285
8,166,95,116.68932775511074,21.68932775511074
9,55,111,133.08384483121054,22.08384483121054
10,192,96,129.1630912436346,33.1630912436346


In [0]:
# Save official test evaluation results
test_evaluation.write.format("delta").mode("overwrite").saveAsTable(
    f"{schema}.gold_test_evaluation"
)

print(
    "Saved evaluation rows:",
    spark.table(f"{schema}.gold_test_evaluation").count()
)

print(f"Test MAE:  {test_mae:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print("Model evaluation saved successfully.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Saved evaluation rows: 100
Test MAE:  20.89
Test RMSE: 28.58
Model evaluation saved successfully.


In [0]:
import mlflow
import mlflow.spark

model_uri = (
    "runs:/1f0d8c6176154f74a2dcc3269f8b3cd3/"
    "random_forest_pipeline"
)

loaded_model = mlflow.spark.load_model(model_uri)

print("Saved model loaded successfully!")
print("Model type:", type(loaded_model).__name__)

Saved model loaded successfully!
Model type: PipelineModel


In [0]:
# Load the existing Gold test features
test_features = spark.table(
    "workspace.predictive_maintenance_rul.gold_test_features"
)

# Generate predictions using the loaded MLflow model
loaded_predictions = loaded_model.transform(test_features)

# Display sample predictions
display(
    loaded_predictions.select(
        "unit_number",
        "cycle",
        "predicted_RUL"
    )
    .orderBy("unit_number", "cycle")
    .limit(10)
)

print("Total prediction rows:", loaded_predictions.count())

unit_number,cycle,predicted_RUL
1,1,156.51407031803316
1,2,152.0841811508738
1,3,152.48514017235237
1,4,151.59971174682278
1,5,154.00150009591417
1,6,160.0618474711556
1,7,154.02318842638434
1,8,151.2659346757777
1,9,158.84218724427444
1,10,153.91734404378658


Total prediction rows: 13096
